In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")

print("Path to dataset files:", path)

In [ ]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns 
import cv2
import tensorflow as tf
from tensorflow import keras

# Read the data

In [ ]:
path ='/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color'
train_ds , test_ds = keras.utils.image_dataset_from_directory(
    path ,
    image_size=(224,224),
    batch_size=32 ,
    seed = 123 ,
    validation_split=.2,
    subset='both'
)

In [ ]:
classes = train_ds.class_names
classes

# Visualize Some Images

In [ ]:
import matplotlib.pyplot as plt

for images, labels in train_ds.take(1):
    plt.figure(figsize=(10,10))
    for i in range(9):
        ax = plt.subplot(3,3,i+1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(classes[labels[i]])
        plt.axis("off")

In [ ]:
image = cv2.imread("/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color/Apple___Apple_scab/01f3deaa-6143-4b6c-9c22-620a46d8be04___FREC_Scab 3112.JPG")

if image is None:
    print("Image not loaded ❌ Check path")
else:
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(6,6))
    plt.imshow(image)
    plt.title('Apple scab')
    plt.axis('off')
    plt.show()

In [ ]:
image = cv2.imread("/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color/Peach___Bacterial_spot/00e6ad4a-5a62-48d7-ac68-9c0b8ec87f5f___Rut._Bact.S 1472.JPG")

if image is None:
    print("Image not loaded ❌ Check path")
else:
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(6,6))
    plt.imshow(image)
    plt.title('Peach Bacterial spot', size=18)
    plt.axis('off')
    plt.show()

In [ ]:


image = cv2.imread("/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color/Tomato___Septoria_leaf_spot/015c2613-fb1c-4f31-88f1-c7e5be9ddc97___JR_Sept.L.S 8431.JPG")

if image is None:
    print("Image not loaded ❌ Check path")
else:
    # Fix color (BGR → RGB)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(6,6))
    plt.imshow(image)
    plt.title('Tomato Septoria leaf spot', size=18)
    plt.axis('off')
    plt.show()

In [ ]:
image = cv2.imread('/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color/Grape___healthy/00e00912-bf75-4cf8-8b7d-ad64b73bea5f___Mt.N.V_HL 6067.JPG')

if image is None:
    print("Image not loaded ❌ Check path")

else:
    image= cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(6,6))
    plt.imshow(image)
    plt.title("healthy grapes leaf",size=18)
    plt.axis('off')
    plt.show()
    
    
    

# Build Neural network [CNN]

In [ ]:
from tensorflow import keras

model = keras.Sequential([
    keras.layers.Rescaling(1./255, input_shape=(224,224,3)),

    keras.layers.Conv2D(32, (3,3), activation='relu'),
    keras.layers.MaxPool2D((2,2)),
    keras.layers.Dropout(0.2),

    keras.layers.Conv2D(64, (3,3), activation='relu'),
    keras.layers.MaxPool2D((2,2)),
    keras.layers.Dropout(0.2),

    keras.layers.Conv2D(64, (3,3), activation='relu'),
    keras.layers.MaxPool2D((2,2)),
    keras.layers.Dropout(0.2),

    keras.layers.Conv2D(64, (3,3), activation='relu'),
    keras.layers.MaxPool2D((2,2)),
    keras.layers.Dropout(0.2),

    keras.layers.Conv2D(128, (3,3), activation='relu'),
    keras.layers.MaxPool2D((2,2)),

    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(64, activation='relu'),

    # ✅ FIXED OUTPUT
    keras.layers.Dense(38, activation='softmax')
])

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']   # ✅ must be list
)

In [ ]:
model.summary()

In [ ]:
history = model.fit(train_ds,epochs=20)


In [ ]:
accuracy = history.history['accuracy']
loss = history.history['loss']


epochs = range(1, len(accuracy)+1)

plt.plot(epochs, accuracy, label='Accuracy')
plt.plot(epochs, loss, label='Loss')

plt.legend()
plt.show()

In [ ]:
model.evaluate(test_ds)

# Test model predictions

In [ ]:
def img_to_pred(image):
  image = image.numpy()
  image = tf.expand_dims(image,0)
  return image

In [ ]:
plt.figure(figsize=(18,18))
for images, labels in test_ds.take(1) : # take the first patch
  for i in range(1,10):
    plt.subplot(3,3,i)
    plt.imshow(images[i].numpy().astype('uint32'))
    plt.axis('off')
    actual = classes[labels[i]]
    predict =classes[np.argmax( model.predict(img_to_pred(images[i])))]
    plt.title(f"actual : {actual}  \n predicted : {predict} ")

In [ ]:
model.save("plant_model.h5") 

In [ ]:
import os

print(os.listdir('/kaggle/working'))